# 🔐 Notebook 3: Privacy-Preserving AI with Local LLMs

**Workshop:** Practical Introduction to AI for Mental Health Applications  
**Estimated time:** 20 minutes  

## Learning Objectives
1. Understand why sending patient data to cloud LLM APIs violates privacy regulations
2. Run Ollama — an open-weight LLM — directly inside this Colab session (no API key required)
3. Apply LLMs to three clinical NLP tasks: summarization, information extraction, de-identification
4. Connect LLM-extracted features to the ML pipeline from Notebook 1

---

> ⚠️ **PRIVACY NOTE**
>
> This notebook runs the LLM **locally within your Colab session** using Ollama.
> Your data never leaves the Colab virtual machine — no external API key needed.
>
> In this workshop we use **synthetic/fictional vignettes only**.
> For real patient data, run Ollama on your own institutional hardware (instructions at the end).

> 🖥️ **IMPORTANT: Colab Runtime**
>
> Before running any cells: **Runtime → Change runtime type → T4 GPU**
> Ollama requires a GPU runtime to run at a useful speed.

## 0. Configuration & Setup

Run these cells first — they install Ollama inside this Colab session and download the model.

In [ ]:
# ============================================================
# CONFIGURATION — Modify this cell if needed
# ============================================================
OLLAMA_MODEL = "llama3.2:1b"  # Fast 1B model for Colab demos; swap for 'llama3.2' (3B) locally

# Detect whether we are running inside Google Colab
try:
    import google.colab  # noqa: F401
    RUNNING_IN_COLAB = True
except ImportError:
    RUNNING_IN_COLAB = False

print(f"Model : {OLLAMA_MODEL}")
print(f"Colab : {RUNNING_IN_COLAB}")
if RUNNING_IN_COLAB:
    print("\n✓ Running in Colab — Ollama will be installed in the next cell.")
    print("  Make sure you selected a T4 GPU runtime (Runtime → Change runtime type).")
else:
    print("\nRunning locally — ensure Ollama is installed and 'ollama serve' is running.")
    print(f"  Pull the model if you haven't: ollama pull {OLLAMA_MODEL}")


In [ ]:
# Install required packages
!pip install requests ollama --quiet

import os
import json
import time
import subprocess
import requests
import ollama
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

print("✓ All imports successful!")


In [ ]:
# Install and start Ollama inside this Colab session
# (If running locally, skip this cell — just ensure 'ollama serve' is running)

if RUNNING_IN_COLAB:
    # Step 1: Install Ollama (ollama.ai URL — confirmed working in Colab)
    print("Installing Ollama...")
    os.system("curl https://ollama.ai/install.sh | sh")

    # Step 2: Set NVIDIA library path before starting the server
    os.environ['LD_LIBRARY_PATH'] = '/usr/lib64-nvidia'
    print("✓ LD_LIBRARY_PATH set for NVIDIA GPU support.")

    # Step 3: Start the server (binary is on PATH after step 1)
    print("Starting Ollama service...")
    subprocess.Popen(
        ['ollama', 'serve'],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    time.sleep(5)

    # Step 4: Pull model (os.system shows live download progress)
    print(f"Pulling model: {OLLAMA_MODEL} ...")
    os.system(f"ollama pull {OLLAMA_MODEL}")
    print(f"✓ {OLLAMA_MODEL} is ready!")

else:
    # Running locally — verify the service is reachable
    try:
        requests.get("http://localhost:11434", timeout=3)
        print("✓ Local Ollama service is reachable.")
    except Exception as e:
        print(f"⚠️  Cannot reach Ollama at localhost:11434 — {e}")
        print("   Start it with: ollama serve")


## 1. Why Local LLMs for Clinical Research?

### The Data Sovereignty Problem

When you use ChatGPT, Claude.ai, or Gemini with patient data, that data:
- Travels over the internet to third-party servers
- May be stored in server logs
- May be used to train future models (depending on API terms)
- Is processed outside your institution's security perimeter

This violates:
- **PIPEDA** (Canada's federal privacy law)
- **PHIPA** (Ontario's provincial health privacy act)
- **HIPAA** (USA health privacy regulation)
- Most institutional data governance agreements

### Open-Weight Models: What Does "Open-Weight" Mean?

| Term | Meaning |
|------|--------|
| Open-weight | Model parameters publicly released — you can run it locally |
| Open-source | Code and training process also released |
| Closed/proprietary | Model runs on company servers only (GPT-4, Claude) |

**Key insight:** Open-weight models (Llama 3, Gemma 3, Mistral) can run on your laptop. The model comes to your data — your data doesn't go to the model.

### 5 Concrete Use Cases for Local LLMs in Mental Health Research

1. **Clinical note summarization** — Condense 3-page discharge summaries into structured bullet points
2. **Structured information extraction** — Extract PHQ-9 scores, diagnoses, medications from free text
3. **De-identification** — Replace PHI (names, dates, locations) with placeholder tokens
4. **Survey coding** — Code open-ended qualitative responses into thematic categories at scale
5. **Synthetic vignette generation** — Create realistic fictional clinical cases for training/testing purposes

## 2. Ollama Setup

Ollama is now running inside this Colab session — no local installation needed for the workshop.
The LLM processes your prompts on the Colab virtual machine; nothing is sent to an external API.

---

### 📖 Running Ollama on your own device (after the workshop)

| OS | Install command |
|----|----------------|
| macOS | `brew install ollama` |
| Linux | `curl -fsSL https://ollama.com/install.sh \| sh` |
| Windows | Download from https://ollama.com/download/windows |

Then in a terminal:
```bash
ollama serve          # Start the background service
ollama pull llama3.2  # Download the 3B model (~2 GB) — higher quality than 1B
ollama pull gemma3:4b # Google's 4B model (~3.3 GB)
ollama pull mistral   # Mistral 7B (~4.1 GB) — highest quality
ollama list           # See installed models
ollama run llama3.2   # Interactive chat
```

**Hardware requirements:**
- 3B models (`llama3.2`): 8 GB RAM minimum
- 7B models (`mistral`): 16 GB RAM + GPU recommended
- Storage: 2–5 GB per model

Browse available models: https://ollama.com/library


In [ ]:
# Verify that the Ollama service is accepting requests
try:
    r = requests.get("http://localhost:11434", timeout=5)
    print("✓ Ollama service is running and reachable.")
    # List available models
    models_resp = requests.get("http://localhost:11434/api/tags", timeout=5)
    models = models_resp.json().get("models", [])
    if models:
        print("Installed models:")
        for m in models:
            size_gb = m.get('size', 0) / 1e9
            print(f"  • {m['name']}  ({size_gb:.1f} GB)")
    else:
        print("No models found — did the pull in the previous cell complete?")
except Exception as e:
    print(f"⚠️  Cannot reach Ollama: {e}")
    print("   In Colab: re-run the setup cell above.")
    print("   Locally: run 'ollama serve' in a terminal.")


## 3. Our Dataset — Synthetic Clinical Vignettes

We'll work with 20 synthetic youth mental health clinical vignettes. These are entirely fictional — programmatically generated to be clinically realistic for youth mental health contexts (ages 12–24).

**⚠️ Important:** These vignettes contain no real patient data. They are generated from clinical knowledge to demonstrate LLM capabilities. Never use these as actual clinical examples.

In [ ]:
# SYNTHETIC CLINICAL VIGNETTES
# These are entirely fictional — generated for demonstration purposes only
# No real patient data is represented here

VIGNETTES_RAW = [
    {
        "patient_id": "SYN-001", "age": 16, "sex": "F",
        "dx_provisional": "MDD", "PHQ9_score": 19, "GAD7_score": 12,
        "presenting_complaint": (
            "16-year-old female referred by school counselor after declining academic performance "
            "over the past 3 months. Reports persistent low mood, anhedonia, and hypersomnia (sleeping "
            "11-12 hours daily). Denies current active suicidal ideation but endorses passive death "
            "wishes ('I wouldn't mind if I didn't wake up'). Social withdrawal from peer group noted. "
            "No prior psychiatric history. Parents report increased irritability at home. Appetite "
            "decreased with approximately 4 kg weight loss over 6 weeks. Currently not on any medications."
        )
    },
    {
        "patient_id": "SYN-002", "age": 14, "sex": "M",
        "dx_provisional": "ADHD", "PHQ9_score": 8, "GAD7_score": 6,
        "presenting_complaint": (
            "14-year-old male referred by pediatrician for behavioral concerns at school. Parents "
            "report long-standing difficulty sustaining attention, frequent interrupting, and inability "
            "to complete homework without significant prompting. Teachers describe him as disruptive "
            "in class. No significant mood symptoms endorsed. Denies anxiety or depressive episodes. "
            "Sleep reported as adequate (8-9 hours). Appetite normal. Currently taking melatonin PRN "
            "for occasional sleep onset difficulty. Family history of ADHD (father). IQ assessment "
            "pending."
        )
    },
    {
        "patient_id": "SYN-003", "age": 22, "sex": "F",
        "dx_provisional": "GAD", "PHQ9_score": 11, "GAD7_score": 18,
        "presenting_complaint": (
            "22-year-old female university student presenting with escalating anxiety over the past "
            "8 months. Reports excessive worry about academic performance, finances, and family health "
            "that she finds difficult to control. Somatic complaints include muscle tension, frequent "
            "headaches, and gastrointestinal upset before exams. Sleep onset difficulties (takes "
            "60-90 minutes to fall asleep). Currently taking sertraline 50mg daily prescribed by "
            "family physician 6 weeks ago with minimal improvement noted to date. Denies panic attacks "
            "or agoraphobia. PHQ-9 elevated consistent with mild-moderate comorbid depressive symptoms."
        )
    },
    {
        "patient_id": "SYN-004", "age": 12, "sex": "M",
        "dx_provisional": "ASD+Anxiety", "PHQ9_score": 7, "GAD7_score": 14,
        "presenting_complaint": (
            "12-year-old male with confirmed ASD diagnosis (age 8) presenting with increased anxiety "
            "symptoms following school transition to middle school. Parents report significant distress "
            "around routine changes, meltdowns 2-3 times weekly, and school refusal on 4 occasions in "
            "the past month. Sensory sensitivities to noise and crowds worsened. Restricted interests "
            "in trains remain stable. Sleep disrupted with multiple nighttime wakings. Currently "
            "receiving behavioral therapy (ABA) twice weekly. Fluoxetine 10mg daily started 4 weeks "
            "ago. No self-harm or suicidal ideation reported."
        )
    },
    {
        "patient_id": "SYN-005", "age": 18, "sex": "M",
        "dx_provisional": "MDD+SIB", "PHQ9_score": 23, "GAD7_score": 9,
        "presenting_complaint": (
            "18-year-old male presenting to emergency department following disclosure of non-suicidal "
            "self-injury (cutting forearms) to school counselor. Reports low mood for approximately "
            "6 months, worsening after parental separation. Endorses hopelessness and worthlessness. "
            "Denies active suicidal ideation with intent or plan at this time but history of passive "
            "ideation. PHQ-9 score 23 consistent with severe depression. No current medications. "
            "Substance use: occasional cannabis (weekly). Safety plan developed. Crisis line numbers "
            "provided. Inpatient admission declined by patient and parents; intensive outpatient "
            "referral made."
        )
    },
    {
        "patient_id": "SYN-006", "age": 15, "sex": "F",
        "dx_provisional": "Anorexia", "PHQ9_score": 14, "GAD7_score": 11,
        "presenting_complaint": (
            "15-year-old female referred by pediatrician due to significant weight loss over 5 months "
            "(BMI now 16.8). Reports restricting food intake due to fear of weight gain and distorted "
            "body image. Denies purging behaviors. Amenorrhea for 3 months. Lanugo noted on physical "
            "exam. Vital signs: HR 52 bpm, BP 90/60. Reports feeling cold constantly. Cognitive "
            "rigidity around food rules noted on mental status exam. Attends competitive dance program "
            "5 days weekly. Minimizes severity of symptoms. Family meal support initiated. Dietitian "
            "referral placed. No current psychiatric medications."
        )
    },
    {
        "patient_id": "SYN-007", "age": 20, "sex": "M",
        "dx_provisional": "FEP", "PHQ9_score": 9, "GAD7_score": 7,
        "presenting_complaint": (
            "20-year-old male brought by family following 3-month history of social withdrawal, "
            "auditory hallucinations (voices commenting on his actions), and persecutory beliefs "
            "(neighbors monitoring him). Academic functioning at university severely impaired. Sleep "
            "erratic. Cannabis use approximately 4-5 times weekly, started age 16. Family history "
            "of schizophrenia (maternal uncle). MSE: blunted affect, thought form tangential, "
            "endorses first-rank symptoms. Risperidone 1mg initiated. EPS monitoring planned. "
            "Referred to Early Psychosis Intervention program. No prior psychiatric treatment."
        )
    },
    {
        "patient_id": "SYN-008", "age": 13, "sex": "F",
        "dx_provisional": "Social Anxiety", "PHQ9_score": 6, "GAD7_score": 15,
        "presenting_complaint": (
            "13-year-old female referred by family physician for school-related anxiety. Reports "
            "intense fear of negative evaluation by peers, avoidance of eating in cafeteria, oral "
            "presentations, and group activities. Symptoms present since age 10 but worsening "
            "significantly in grade 8. No significant depressive symptoms. Sleep adequate. Appetite "
            "normal. Selective mutism with unfamiliar adults noted in session. Currently in CBT "
            "with private therapist (6 sessions completed). Parents report significant family "
            "accommodation of avoidance behaviors. No medications currently prescribed."
        )
    },
    {
        "patient_id": "SYN-009", "age": 24, "sex": "F",
        "dx_provisional": "PTSD", "PHQ9_score": 16, "GAD7_score": 13,
        "presenting_complaint": (
            "24-year-old female presenting with nightmares, hypervigilance, and intrusive memories "
            "following traumatic event (assault) 14 months ago. Avoids public transit and crowds. "
            "Reports emotional numbing and estrangement from friends. Concentration significantly "
            "impaired affecting work performance. Started venlafaxine 75mg 3 months ago with partial "
            "response. Currently waiting for EMDR therapy (6-month waitlist). PCL-5 score: 54 "
            "(severe range). PHQ-9 elevated consistent with comorbid depression. Denies current "
            "suicidal ideation. Supportive partner. Housing stable."
        )
    },
    {
        "patient_id": "SYN-010", "age": 17, "sex": "M",
        "dx_provisional": "Bipolar II", "PHQ9_score": 12, "GAD7_score": 5,
        "presenting_complaint": (
            "17-year-old male presenting during a depressive episode. History includes two previous "
            "hypomanic periods (decreased sleep need to 3-4 hours, increased goal-directed activity, "
            "pressured speech, spending spree) each lasting approximately 5-7 days without hospitalization. "
            "Current episode: low mood 3 weeks, anhedonia, hypersomnia, difficulty concentrating. "
            "Family history of bipolar disorder (mother, treated with lithium). Previously prescribed "
            "escitalopram by family physician which triggered hypomania 8 months ago. Now on quetiapine "
            "50mg QHS for 2 weeks. Referred to youth mood disorders program for specialist assessment."
        )
    },
    {
        "patient_id": "SYN-011", "age": 19, "sex": "F",
        "dx_provisional": "BPD", "PHQ9_score": 17, "GAD7_score": 14,
        "presenting_complaint": (
            "19-year-old female with longstanding emotional dysregulation presenting following breakup "
            "with partner of 2 years. Reports intense abandonment fear, chronic emptiness, and identity "
            "disturbance. History of 3 previous emergency department visits for self-harm (superficial "
            "cutting). Relationships described as intense and unstable. Dissociative episodes reported "
            "when emotionally overwhelmed. Currently in dialectical behavior therapy skills group "
            "(8 weeks completed). Olanzapine 2.5mg PRN for acute dysregulation episodes. Denies "
            "current active suicidal ideation with plan but endorses chronic passive ideation."
        )
    },
    {
        "patient_id": "SYN-012", "age": 15, "sex": "M",
        "dx_provisional": "OCD", "PHQ9_score": 10, "GAD7_score": 16,
        "presenting_complaint": (
            "15-year-old male presenting with intrusive contamination obsessions and washing compulsions "
            "occupying 3-4 hours daily. Rituals include handwashing 30+ times daily (hands chapped "
            "and bleeding), avoidance of doorknobs and school bathrooms. Significant functional "
            "impairment — late to class daily due to rituals. Y-BOCS score: 28 (severe). No ASD "
            "features. No mood disorder. Family history of OCD (aunt). No prior treatment. Starting "
            "ERP with specialized OCD therapist next week. Fluvoxamine 50mg started today. Parent "
            "psychoeducation provided regarding accommodation reduction."
        )
    },
    {
        "patient_id": "SYN-013", "age": 21, "sex": "M",
        "dx_provisional": "AUD", "PHQ9_score": 13, "GAD7_score": 8,
        "presenting_complaint": (
            "21-year-old male presenting for alcohol use concerns. Reports drinking 8-10 drinks "
            "4-5 nights weekly since first year of university. Continues drinking despite failing "
            "two courses, DUI charge 3 months ago, and relationship breakdown. Morning drinking "
            "started 2 months ago. AUDIT score: 24. Reports anxiety when not drinking. Denies "
            "seizure history. CAGE: 3/4. Significant family history of alcohol use disorder "
            "(both parents). Currently unemployed. Referred to addiction medicine. Thiamine "
            "supplementation initiated. Collaborative care plan with recovery coach established. "
            "Naltrexone 50mg considered pending medical clearance."
        )
    },
    {
        "patient_id": "SYN-014", "age": 16, "sex": "F",
        "dx_provisional": "MDD", "PHQ9_score": 20, "GAD7_score": 10,
        "presenting_complaint": (
            "16-year-old female with Type 1 diabetes presenting with worsening depression over "
            "6 months concurrent with significant HbA1c deterioration (from 7.2% to 9.8%). Reports "
            "intentional insulin omission as a method of self-harm ('diabulimia'). Weight loss of "
            "7 kg over 4 months despite adequate caloric intake. PHQ-9: 20. Hopelessness endorsed. "
            "No active suicidal ideation. Pediatric endocrinology co-management established. "
            "Coordinated care meeting with diabetes educator, dietitian, and psychiatry team "
            "scheduled. SSRI initiation deferred pending further assessment. School accommodation "
            "letter requested."
        )
    },
    {
        "patient_id": "SYN-015", "age": 23, "sex": "M",
        "dx_provisional": "ADHD+MDD", "PHQ9_score": 15, "GAD7_score": 9,
        "presenting_complaint": (
            "23-year-old male with childhood ADHD diagnosis (on methylphenidate from ages 9-17) "
            "presenting with recurrent depressive episodes and requests to restart ADHD treatment. "
            "Reports poor work performance, procrastination, time blindness, and emotional dysregulation. "
            "Current PHQ-9: 15 (moderate depression). Functional impairment in new professional role. "
            "No current medications. No substance use. Physical exam and vitals normal. Neuropsychological "
            "testing requested to clarify ADHD vs. executive dysfunction secondary to depression. "
            "Supportive counseling initiated. Medication options (SNRI vs. stimulant sequencing) "
            "discussed with patient."
        )
    },
    {
        "patient_id": "SYN-016", "age": 12, "sex": "F",
        "dx_provisional": "Selective Mutism", "PHQ9_score": 5, "GAD7_score": 13,
        "presenting_complaint": (
            "12-year-old female referred by school for selective mutism present since kindergarten. "
            "Speaks freely at home and with close family but completely non-verbal at school and with "
            "unfamiliar adults. Recently started middle school with new teachers and peer group — "
            "symptoms worsened. Communicates via writing and gestures in clinical setting. No ASD "
            "features on developmental history. Avoidant temperament noted since infancy. Parents "
            "describe significant anxiety in novel situations. Previously received play therapy (no "
            "improvement). Referred to selective mutism specialist for behavioral intervention. "
            "School support plan developed with education team."
        )
    },
    {
        "patient_id": "SYN-017", "age": 18, "sex": "F",
        "dx_provisional": "GAD+Insomnia", "PHQ9_score": 9, "GAD7_score": 16,
        "presenting_complaint": (
            "18-year-old female presenting with chronic insomnia (onset 2 years ago) and generalized "
            "anxiety. Reports sleep onset latency of 90-120 minutes, multiple nighttime wakings, and "
            "unrefreshing sleep despite 8-9 hours in bed. Daytime fatigue significantly impacts first "
            "year of college. Anxiety content: health, academic performance, future career. ISI score: "
            "22 (severe insomnia). Caffeine intake: 4-5 coffees daily to manage fatigue. Referred for "
            "CBT-I (cognitive behavioral therapy for insomnia). Sleep hygiene psychoeducation provided. "
            "Melatonin 3mg QHS trialed without benefit. Quetiapine discussed but deferred given young age."
        )
    },
    {
        "patient_id": "SYN-018", "age": 14, "sex": "M",
        "dx_provisional": "Conduct Disorder", "PHQ9_score": 7, "GAD7_score": 4,
        "presenting_complaint": (
            "14-year-old male referred from youth justice following assault charge at school. "
            "History of aggression, property destruction, and truancy since age 11. No consistent "
            "school placement for past 2 years. Cannabis use daily since age 12. Multiple "
            "placements in child protection care. Parents report inability to manage behaviors at "
            "home. CBCL externalizing T-score: 78. Rule out ADHD (neuropsychological assessment "
            "planned). Mood appears reactive/dysphoric rather than sustained. Trauma history "
            "significant (witnessed domestic violence, physical abuse by stepfather). Trauma-informed "
            "care model recommended. Community youth worker assigned. Multisystemic therapy referral made."
        )
    },
    {
        "patient_id": "SYN-019", "age": 22, "sex": "F",
        "dx_provisional": "Panic Disorder", "PHQ9_score": 10, "GAD7_score": 17,
        "presenting_complaint": (
            "22-year-old female presenting with recurrent unexpected panic attacks (3-5 weekly "
            "for past 4 months). Episodes: palpitations, chest tightness, derealization, fear of "
            "dying lasting 8-15 minutes. Emergency department visit last month (ECG and troponins "
            "normal). Now avoids public transit, shopping malls, and exercise due to fear of triggering "
            "attack. Agoraphobic avoidance moderate severity. GAD-7: 17. ASI-3: 36. No prior "
            "psychiatric treatment. Fluoxetine 10mg started today with plan to titrate. Referred "
            "for CBT with interoceptive exposure. Psychoeducation about panic cycle provided. "
            "Safety behaviors identified and challenge hierarchy developed."
        )
    },
    {
        "patient_id": "SYN-020", "age": 17, "sex": "M",
        "dx_provisional": "ASD+ADHD", "PHQ9_score": 11, "GAD7_score": 8,
        "presenting_complaint": (
            "17-year-old male with recently confirmed ASD (level 1) and ADHD-combined type presenting "
            "for medication review. Currently on lisdexamfetamine 40mg daily with partial response — "
            "academic performance improved but emotional dysregulation (low frustration tolerance, "
            "meltdowns) remains problematic. Parents request additional support for transition planning "
            "(high school graduation in 6 months, considering college). Executive function assessment "
            "completed (significant deficits in planning, flexibility). Social skills group completed "
            "last year (minimal generalization). Occupational therapy assessment for workplace "
            "accommodations requested. Lisdexamfetamine dose increase to 50mg discussed."
        )
    }
]

vignettes_df = pd.DataFrame(VIGNETTES_RAW)
print(f"Dataset: {len(vignettes_df)} synthetic patient vignettes")
print(f"Age range: {vignettes_df['age'].min()}-{vignettes_df['age'].max()} years")
print(f"PHQ-9 range: {vignettes_df['PHQ9_score'].min()}-{vignettes_df['PHQ9_score'].max()}")
print(f"\nDiagnoses:")
print(vignettes_df['dx_provisional'].value_counts())
print(f"\nFirst 3 vignettes:")
print(vignettes_df[['patient_id', 'age', 'sex', 'dx_provisional', 'PHQ9_score']].head(3).to_string())

## 4. Task 1 — Clinical Note Summarization

**Clinical context:** Clinicians often receive lengthy referral letters, discharge summaries, or consultation notes. Summarizing these efficiently while preserving key information is a high-value task.

**What we'll do:** Ask the LLM to summarize each vignette into 3 structured bullet points.

In [ ]:
def call_llm(prompt, system_prompt, max_tokens=400):
    """Call the local Ollama server via the ollama Python client."""
    try:
        response = ollama.chat(
            model=OLLAMA_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": prompt},
            ],
            options={"num_predict": max_tokens}
        )
        return response['message']['content']
    except Exception as e:
        return f"[Ollama error: {e}]"


# System prompt for summarization
SUMMARIZE_SYSTEM = """You are a clinical documentation assistant supporting a child and adolescent psychiatry team.
Summarize the following clinical note in exactly 3 bullet points:
- Primary presenting complaint and key symptoms
- Key risk factors or safety considerations identified
- Suggested immediate follow-up priorities

Use clinical language. Be concise. Each bullet point should be 1-2 sentences maximum."""


def summarize_vignette(text, patient_id):
    """Summarize a clinical vignette into structured bullet points."""
    prompt = f"Patient ID: {patient_id}\n\nClinical Note:\n{text}"
    return call_llm(prompt, SUMMARIZE_SYSTEM, max_tokens=300)


# Test on first 2 vignettes
print("=== Testing Clinical Note Summarization ===\n")
for i in range(min(2, len(vignettes_df))):
    row = vignettes_df.iloc[i]
    print(f"Patient {row['patient_id']} | Age {row['age']} | {row['dx_provisional']}")
    print(f"PHQ-9: {row['PHQ9_score']} | GAD-7: {row['GAD7_score']}")
    print("\nOriginal note (excerpt):", row['presenting_complaint'][:200] + "...")
    print("\nLLM Summary:")
    summary = summarize_vignette(row['presenting_complaint'], row['patient_id'])
    print(summary)
    print("\n" + "="*60 + "\n")


> 🔧 **EXERCISE:** Modify the system prompt to ask the model to also assign a risk level (low/medium/high) at the end of the summary.
>
> **Hint:** Add to `SUMMARIZE_SYSTEM`: `"- Overall risk level: [low/medium/high] with one-sentence justification"`. Does the model's risk assessment match the PHQ-9 scores?

## 5. Task 2 — Structured Information Extraction

**Clinical context:** Research databases need structured data (diagnosis codes, medication names, scores), but clinical notes are unstructured text. LLMs can extract structured fields at scale.

**What we'll do:** Ask the LLM to extract key clinical fields as JSON.

In [ ]:
EXTRACT_SYSTEM = """You are a clinical data extraction assistant.
Extract the following information from the clinical note and return ONLY valid JSON — no other text, no explanation, no markdown.

Required JSON structure:
{
  "age_mentioned": <integer or null>,
  "primary_diagnosis_mentioned": <string or null>,
  "risk_level": <"low", "medium", "high", or "unknown">,
  "medications_mentioned": [<list of medication name strings, empty list if none>],
  "key_symptoms": [<list of up to 3 symptom strings>],
  "safety_concern_present": <true or false>
}

Return ONLY the JSON object. No other text."""


def extract_structured_info(text, patient_id="unknown"):
    """Extract structured fields from a clinical note. Returns a dict."""
    prompt = f"Clinical Note:\n{text}"
    raw_response = call_llm(prompt, EXTRACT_SYSTEM, max_tokens=300)
    
    # Parse JSON — handle common LLM formatting issues
    try:
        # Strip markdown code blocks if present
        cleaned = raw_response.strip()
        if cleaned.startswith("```"):
            cleaned = "\n".join(cleaned.split("\n")[1:-1])
        return json.loads(cleaned)
    except json.JSONDecodeError:
        # Return a fallback structure if parsing fails
        print(f"  \u26a0\ufe0f  JSON parse failed for {patient_id}. Raw: {raw_response[:100]}...")
        return {
            "age_mentioned": None, "primary_diagnosis_mentioned": None,
            "risk_level": "unknown", "medications_mentioned": [],
            "key_symptoms": [], "safety_concern_present": False
        }


# Apply to first 5 vignettes for demo (limit API calls during workshop)
print("=== Structured Information Extraction ===\n")
extractions = []
for i in range(min(5, len(vignettes_df))):
    row = vignettes_df.iloc[i]
    extracted = extract_structured_info(row['presenting_complaint'], row['patient_id'])
    extracted['patient_id'] = row['patient_id']  # Add patient ID for joining
    extractions.append(extracted)
    print(f"Patient {row['patient_id']}: {extracted}")

# Convert to DataFrame
structured_df = pd.DataFrame(extractions)
print("\n=== Extracted Structured Data ===")
print(structured_df[['patient_id', 'primary_diagnosis_mentioned', 'risk_level', 
                       'safety_concern_present']].to_string())

> 💬 **DISCUSSION POINT:** This is the bridge between unstructured EHR text and the ML pipeline from Notebook 1. The extracted structured fields (risk level, safety concern, symptoms) become features that a classifier can learn from — without anyone manually coding thousands of notes.
>
> What accuracy would you need from this extraction step before you'd trust it in a research pipeline? 90%? 95%? How would you validate it?

## 6. Task 3 — De-identification

**Clinical context:** Before sharing data across sites, publishing case studies, or training models, identifying information must be removed. Manual de-id is slow; LLMs can do it at scale.

**⚠️ Important caveat:** LLM-based de-identification is NOT 100% reliable. For production use, always validate against ground truth annotations and use specialized tools like Microsoft Presidio or the NLP de-id models from PhysioNet.

In [ ]:
DEID_SYSTEM = """You are a clinical data de-identification assistant.
De-identify the clinical note by replacing all identifying information with placeholder tokens:
- Patient names → [NAME]
- Clinician names → [CLINICIAN]
- Dates (DOB, appointment dates, specific dates) → [DATE]
- Hospitals, clinics, locations → [LOCATION]
- Phone numbers → [PHONE]
- Email addresses → [EMAIL]
- Health card / ID numbers → [ID]
- Any other potentially identifying information → [REDACTED]

Return ONLY the de-identified text with no other commentary."""


def deidentify_note(text):
    """De-identify a clinical note by replacing PHI with placeholder tokens."""
    return call_llm(text, DEID_SYSTEM, max_tokens=500)


# Demo with a sample that has seeded fake PHI
sample_with_phi = (
    "Patient Emily Chen (DOB March 15, 2008) was referred by Dr. Sarah Patel from "
    "SickKids Hospital Toronto on January 12, 2024. Mother can be reached at "
    "416-555-0123. Patient health card number: 1234-567-890. Emily presented to "
    "the Sunnybrook Mental Health Centre for assessment of depression following "
    "discharge from Oshawa General on December 3, 2023."
)

print("=== De-identification Demo ===\n")
print("ORIGINAL TEXT:")
print(sample_with_phi)
print("\nDE-IDENTIFIED TEXT:")
deidentified = deidentify_note(sample_with_phi)
print(deidentified)

print("\n\n--- Now testing on a synthetic vignette ---")
vignette_deid = deidentify_note(vignettes_df.iloc[0]['presenting_complaint'])
print("De-identified vignette (should show no changes — our vignettes are already de-identified):")
print(vignette_deid[:300] + "...")

> 🔧 **EXERCISE:** Add a phone number and street address to the sample text above. Does the model successfully redact them?
>
> **Hint:** Modify `sample_with_phi` to add text like: `"She lives at 42 Maple Street, Toronto, ON M5A 1A1. Contact: emily.chen@gmail.com"`. Re-run the de-identification. What does the model miss?
>
> 💬 **DISCUSSION POINT:** LLMs are surprisingly good at de-identification but not perfect. Rare identifiers (unique job titles, unusual names) may slip through. For production systems, always use validated de-identification tools AND human review of a random sample.

## 7. Connecting LLM Outputs to an ML Pipeline

**The full clinical AI workflow:**
```
Unstructured clinical notes → LLM extraction → Structured features → ML classifier
```

This is exactly how real clinical NLP pipelines work. Let's build the final step: using LLM-extracted features to predict which patients have high PHQ-9 scores (>= 15, indicating moderate-severe depression).

In [ ]:
# For this demo, we use ALL 20 vignettes with a simulated extraction
# (Since calling the API 20 times would be slow/costly, we simulate realistic extractions)
# In real use: extractions = vignettes_df['presenting_complaint'].apply(extract_structured_info)

# Simulate realistic LLM extractions based on the vignette content
np.random.seed(42)
simulated_extractions = []
for _, row in vignettes_df.iterrows():
    # Simulate risk level — correlated with PHQ-9 (as a good LLM extraction would be)
    phq = row['PHQ9_score']
    if phq >= 18:
        risk = "high"
    elif phq >= 12:
        risk = "medium"  
    else:
        risk = np.random.choice(["low", "medium"], p=[0.7, 0.3])
    
    # Simulate safety concern detection
    safety = row['dx_provisional'] in ['MDD+SIB', 'BPD', 'Anorexia', 'FEP']
    
    # Simulate medication detection
    has_meds = np.random.choice([True, False], p=[0.6, 0.4])
    
    simulated_extractions.append({
        'patient_id': row['patient_id'],
        'risk_level': risk,
        'safety_concern_present': safety,
        'has_medications': has_meds
    })

extraction_df = pd.DataFrame(simulated_extractions)

# Join with original PHQ-9 and GAD-7 scores
analysis_df = vignettes_df[['patient_id', 'PHQ9_score', 'GAD7_score', 'age', 'sex']].merge(
    extraction_df, on='patient_id'
)

print("Analysis-ready DataFrame (combining LLM features + structured scores):")
print(analysis_df.to_string())

In [ ]:
# Build a simple classifier: predict high PHQ-9 (>= 15) from LLM-extracted features
analysis_df['high_phq9'] = (analysis_df['PHQ9_score'] >= 15).astype(int)

# Encode categorical features
risk_encoder = LabelEncoder()
analysis_df['risk_level_encoded'] = risk_encoder.fit_transform(analysis_df['risk_level'])
sex_encoder = LabelEncoder()
analysis_df['sex_encoded'] = sex_encoder.fit_transform(analysis_df['sex'])

# Features: LLM-extracted risk level + safety concern + GAD-7 + age + sex
feature_cols_llm = ['risk_level_encoded', 'safety_concern_present', 
                     'has_medications', 'GAD7_score', 'age', 'sex_encoded']
X_llm = analysis_df[feature_cols_llm].astype(float)
y_llm = analysis_df['high_phq9']

# Split (small dataset — stratify to preserve class balance)
if y_llm.sum() >= 4 and (1 - y_llm).sum() >= 4:
    X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
        X_llm, y_llm, test_size=0.3, random_state=42, stratify=y_llm
    )
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_train_l, y_train_l)
    y_pred_l = clf.predict(X_test_l)
    
    print("=== LLM Features \u2192 ML Classifier ===")
    print(f"Predicting high PHQ-9 (>= 15) from LLM-extracted features\n")
    print(classification_report(y_test_l, y_pred_l, target_names=['Normal PHQ-9', 'High PHQ-9']))
    
    # Feature importance (logistic regression coefficients)
    coef_df = pd.DataFrame({'Feature': feature_cols_llm, 'Coefficient': clf.coef_[0]})
    coef_df = coef_df.reindex(coef_df['Coefficient'].abs().sort_values(ascending=False).index)
    
    fig, ax = plt.subplots(figsize=(9, 5))
    colors = ['#B93680' if c > 0 else '#2563EB' for c in coef_df['Coefficient']]
    ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, alpha=0.8)
    ax.axvline(0, color='black', linewidth=1)
    ax.set_xlabel('Logistic Regression Coefficient', fontsize=12)
    ax.set_title('Which LLM-Extracted Features Predict High PHQ-9?\n(Pink=increases risk, Blue=decreases)', 
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('llm_feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Note: Dataset too small for reliable train/test split. In real use, n >> 20.")

print("\n\ud83d\udcac This is the full pipeline: EHR text \u2192 LLM \u2192 structured features \u2192 ML classifier")
print("   In a real study: apply to hundreds/thousands of notes, validate extractions, then train.")

## 8. Federated Learning — The Next Frontier (Conceptual)

### The Problem Federated Learning Solves

Multi-site research requires more data than any single institution has. But sharing patient data across sites is often legally and ethically impossible. Federated learning offers a solution.

### How It Works

```
                    ┌─────────────────────┐
                    │   Central Server    │
                    │  (coordinates only) │
                    └──────┬──────┬───────┘
                           │      │
              ┌────────────▼──┐ ┌─▼────────────┐
              │  Hospital A   │ │  Hospital B   │
              │  Local data   │ │  Local data   │
              │  Local train  │ │  Local train  │
              └───────┬───────┘ └───────┬───────┘
                      │                 │
              Model weights only (no patient data)
                      │                 │
                      └────────┬────────┘
                               ▼
                    Updated global model

Patient data NEVER leaves each hospital.
Only model parameters (billions of numbers) are shared.
```

### Real-World Frameworks
- **NVIDIA FLARE** — Production-grade federated learning for healthcare
- **PySyft (OpenMined)** — Privacy-preserving ML with differential privacy
- **HealthChain** — Blockchain-based federated learning for EHR data
- **FLOWER (flwr)** — Flexible federated learning framework in Python

### Why It Matters for Youth Mental Health
ASD, eating disorders, and first-episode psychosis are rare — no single site has enough cases for robust ML models. Federated learning enables collaboration across CHEO, SickKids, BC Children's, Montreal Children's without centralizing data.

### Limitations & Open Research Questions
- Model gradients can sometimes leak training data (gradient inversion attacks)
- Communication overhead can be significant with many sites
- Heterogeneous data distributions across sites complicate aggregation
- Regulatory frameworks for federated learning data governance are still evolving

## 9. Summary & Next Steps

### What we covered across all 3 notebooks:

| Notebook | Topic | Key Tool |
|----------|-------|----------|
| 1 | ML Foundations | scikit-learn |
| 2 | Explainability | SHAP |
| 3 | Privacy-Preserving LLMs | Ollama (local / Colab) |

### How to adapt these notebooks to your own data:
1. **Replace the synthetic dataset** — In Notebook 1, swap `generate_abide_dataset()` for your data loading function
2. **Adjust feature names** — Update `feature_cols` to match your variable names
3. **Tune preprocessing** — Adjust imputation strategy and scaling for your data distributions
4. **Validate LLM extractions** — Always manually review a 10–20 % random sample of LLM outputs

### Running Ollama on your own device after this workshop

The same open-weight model we just used in Colab can run entirely on your own laptop or server —
no internet connection, no API key, no data leaving your machine.

**macOS**
```bash
brew install ollama
ollama serve          # Start the background service
ollama pull llama3.2  # Download 3B model (~2 GB)
```

**Windows** — Download installer: https://ollama.com/download/windows

**Linux**
```bash
curl -fsSL https://ollama.com/install.sh | sh
ollama pull llama3.2
```

To use this notebook locally, change `OLLAMA_MODEL` in Cell 2 to `"llama3.2"` (better quality)
and skip the Colab setup cell.

**Hardware requirements:**
- 3B models (`llama3.2`): 8 GB RAM minimum
- 7B models (`mistral`, `llama3.1:7b`): 16 GB RAM + GPU recommended
- Storage: 2–5 GB per model

### Community Resources
- 🌐 fast.ai — Free practical ML course: https://fast.ai
- 🤗 Hugging Face — Model hub and tutorials: https://huggingface.co
- 📄 Ollama model library: https://ollama.com/library
- 🏥 Machine Learning for Healthcare (MLHC) conference papers
- 🇨🇦 CIFAR Pan-Canadian AI Strategy resources

### GitHub Repository
These notebooks will be updated and extended after the workshop.
→ **[GITHUB_REPO_URL]** — Star the repo to get updates!

Thank you for participating! 🎉
